In [2]:
pwd

'c:\\Users\\adeka\\OneDrive\\Documents\\credit-scoring-Project\\research'

In [3]:
import os
os.chdir(r"C:\Users\adeka\OneDrive\Documents\credit-scoring-Project")

print(os.getcwd())

C:\Users\adeka\OneDrive\Documents\credit-scoring-Project


In [ ]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    train_data_path: Path
    test_data_path: Path
    woe_bins_path: Path
    model_name: str
    C: float
    max_iter: int
    solver: str
    class_weight: str
    target_column: str

In [5]:
from src.mlProject.constants import *
from src.mlProject.utils.common import read_yaml, create_directories

In [6]:
class ConfigurationManager:
    def __init__(
        self,
        config_file_path: Path = CONFIG_FILE_PATH,
        params_file_path: Path = PARAMS_FILE_PATH,
        schema_file_path: Path = SCHEMA_FILE_PATH
    ):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        self.schema = read_yaml(schema_file_path)

        create_directories([self.config.artifacts_root])
    
    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.LogisticRegression

        create_directories([config.root_dir])

        return ModelTrainerConfig(
            root_dir=config.root_dir,
            train_data_path=config.train_data_path,
            test_data_path=config.test_data_path,
            woe_bins_path=config.woe_bins_path,
            model_name=config.model_name,
            C=params.C,
            max_iter=params.max_iter,
            solver=params.solver,
            class_weight=params.class_weight,
            target_column=self.schema.TARGET_COLUMN
        )

In [8]:
import joblib
import json
import numpy as np
import pandas as pd
import scorecardpy as sc                                    # ← tambah
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score                  # ← tambah

from src.mlProject.entity.config_entity import ModelTrainerConfig
from src.mlProject.constants import TARGET_COLUMN
from src.mlProject.logging import logger

In [9]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def _load_data(self):
        train = pd.read_csv(self.config.train_data_path)
        test  = pd.read_csv(self.config.test_data_path)
        logger.info(f"Train: {train.shape} | Test: {test.shape}")
        return train, test

    def _check_class_distribution(self, y: pd.Series, label: str):
        dist = y.value_counts(normalize=True).round(4) * 100
        logger.info(f"Class distribution [{label}]:")
        for cls, pct in dist.items():
            logger.info(f"  Kelas {cls}: {pct:.2f}%")

        if dist.min() < 20 and self.config.class_weight is not None:
            logger.info("  ⚠ Imbalance terdeteksi — class_weight aktif")

    def _log_coefficients(self, model: LogisticRegression, features: list):
        coef_df = pd.DataFrame({
            "feature": features,
            "coefficient": model.coef_[0],
            "odds_ratio": np.exp(model.coef_[0]),
        }).sort_values("coefficient", ascending=False)

        logger.info("Top 5 bad-risk features:")
        for _, row in coef_df.head(5).iterrows():
            logger.info(
                f"  {row['feature']:35s} "
                f"coef={row['coefficient']:+.4f}  OR={row['odds_ratio']:.4f}"
            )
        return coef_df

    def initiate_model_training(self):
        train, test = self._load_data()

        if TARGET_COLUMN not in train.columns:
            raise ValueError(f"TARGET '{TARGET_COLUMN}' tidak ada di train")
        if train[TARGET_COLUMN].isna().all():
            raise ValueError("Target kosong di train")

        feature_cols = [c for c in train.columns if c != TARGET_COLUMN]
        X_train, y_train = train[feature_cols], train[TARGET_COLUMN]
        X_test,  y_test  = test[feature_cols],  test[TARGET_COLUMN]

        logger.info(f"WOE features: {len(feature_cols)}")
        self._check_class_distribution(y_train, "train")

        logger.info("Training Logistic Regression ...")
        model = LogisticRegression(
            C=self.config.C,
            max_iter=self.config.max_iter,
            solver=self.config.solver,
            class_weight=self.config.class_weight,
            random_state=42,
        )
        model.fit(X_train, y_train)
        logger.info(f"Training selesai ✅ iterasi={model.n_iter_[0]}")

        auc_train = roc_auc_score(y_train, model.predict_proba(X_train)[:, 1])
        auc_test  = roc_auc_score(y_test,  model.predict_proba(X_test)[:, 1])
        logger.info(f"AUC Train={auc_train:.4f} | AUC Test={auc_test:.4f} | Gap={auc_train-auc_test:.4f}")

        coef_df = self._log_coefficients(model, feature_cols)

        root = Path(self.config.root_dir)
        root.mkdir(parents=True, exist_ok=True)

        joblib.dump(model, root / self.config.model_name)
        logger.info(f"Model    : {root / self.config.model_name}")

        with open(root / "feature_names.json", "w") as f:
            json.dump(feature_cols, f, indent=2)

        coef_df.to_csv(root / "coefficients.csv", index=False)

        try:
            bins = joblib.load(self.config.woe_bins_path)
            card = sc.scorecard(bins, model, feature_cols)
            joblib.dump(card, root / "scorecard.pkl")
            logger.info(f"Scorecard: {root / 'scorecard.pkl'}")
        except Exception as e:
            logger.info(f"Scorecard tidak dapat dibuat: {e}")

        return {
            "model_path": str(root / self.config.model_name),
            "n_features": len(feature_cols),
            "n_iter":     int(model.n_iter_[0]),
            "auc_train":  round(float(auc_train), 4),
            "auc_test":   round(float(auc_test),  4),
        }

In [10]:
from src.mlProject.config.configuration import ConfigurationManager
from src.mlProject.components.model_trainer import ModelTrainer
from src.mlProject.logging import logger

STAGE_NAME = "Model Trainer Stage"


class ModelTrainerTrainingPipeline:
    def __init__(self):
        pass

    def main(self):
        config = ConfigurationManager()
        trainer_config = config.get_model_trainer_config()
        trainer = ModelTrainer(config=trainer_config)
        return trainer.initiate_model_training()


if __name__ == "__main__":
    try:
        logger.info(f">>>>>> stage {STAGE_NAME} started <<<<<<")
        pipeline = ModelTrainerTrainingPipeline()
        metrics = pipeline.main()
        logger.info(f">>>>>> stage {STAGE_NAME} completed <<<<<<\n\nx==========x")
    except Exception as e:
        logger.exception(e)
        raise e

[2026-06-06 04:45:08,257: INFO: 3392433177]
[2026-06-06 04:45:08,266: INFO: common]
[2026-06-06 04:45:08,275: INFO: common]
[2026-06-06 04:45:08,281: INFO: common]
[2026-06-06 04:45:08,288: INFO: common]
[2026-06-06 04:45:08,291: INFO: common]
[2026-06-06 04:45:10,993: INFO: model_trainer]
[2026-06-06 04:45:11,126: INFO: model_trainer]
[2026-06-06 04:45:11,144: INFO: model_trainer]
[2026-06-06 04:45:11,146: INFO: model_trainer]
[2026-06-06 04:45:11,149: INFO: model_trainer]
[2026-06-06 04:45:11,157: INFO: model_trainer]
[2026-06-06 04:45:12,332: INFO: model_trainer]
[2026-06-06 04:45:12,581: INFO: model_trainer]
[2026-06-06 04:45:12,591: INFO: model_trainer]
[2026-06-06 04:45:12,594: INFO: model_trainer]
[2026-06-06 04:45:12,596: INFO: model_trainer]
[2026-06-06 04:45:12,598: INFO: model_trainer]
[2026-06-06 04:45:12,600: INFO: model_trainer]
[2026-06-06 04:45:12,606: INFO: model_trainer]
[2026-06-06 04:45:12,611: INFO: model_trainer]
[2026-06-06 04:45:13,374: INFO: model_trainer]
[202

In [11]:
STAGE_NAME = "Model Trainer Stage"
try:
    logger.info(f">>>>>> stage {STAGE_NAME} started <<<<<<")
    ModelTrainerTrainingPipeline().main()
    logger.info(f">>>>>> stage {STAGE_NAME} completed <<<<<<\n\nx==========x")
except Exception as e:
    logger.exception(e)
    raise e

[2026-06-06 04:45:20,870: INFO: 1055380405]
[2026-06-06 04:45:20,876: INFO: common]
[2026-06-06 04:45:20,879: INFO: common]
[2026-06-06 04:45:20,885: INFO: common]
[2026-06-06 04:45:20,888: INFO: common]
[2026-06-06 04:45:20,890: INFO: common]
[2026-06-06 04:45:23,083: INFO: model_trainer]
[2026-06-06 04:45:23,099: INFO: model_trainer]
[2026-06-06 04:45:23,106: INFO: model_trainer]
[2026-06-06 04:45:23,108: INFO: model_trainer]
[2026-06-06 04:45:23,110: INFO: model_trainer]
[2026-06-06 04:45:23,113: INFO: model_trainer]
[2026-06-06 04:45:23,506: INFO: model_trainer]
[2026-06-06 04:45:23,642: INFO: model_trainer]
[2026-06-06 04:45:23,645: INFO: model_trainer]
[2026-06-06 04:45:23,647: INFO: model_trainer]
[2026-06-06 04:45:23,649: INFO: model_trainer]
[2026-06-06 04:45:23,681: INFO: model_trainer]
[2026-06-06 04:45:23,683: INFO: model_trainer]
[2026-06-06 04:45:23,689: INFO: model_trainer]
[2026-06-06 04:45:23,698: INFO: model_trainer]
[2026-06-06 04:45:24,206: INFO: model_trainer]
[202